## VLM multiview which_shape_next_cam0

- VLM is **gemini-3.8-flash (processing mode: 'agentic')** 
- I made a 4-sided rectangular paper prism with a unique shape on each side. 
- Cam0 is birds eye only sees object top AND cam1 POV view but level with object on table so it can see only one side at a time.
10 vids where i just rotate the cube consistently and ask the model to tell me at end of video which shape it thinks will appear on cam0 with one more turn?
  - which_shape_next_cam0**_a** – 11 vids rotating clockwise. So CURRENT SHAPE on cam1 will be NEXT SHAPE ON cam0. If fewer than 4 rotations made, model cannot predict next shape using cam0 alone, it needs cam1 as well.
  - which_shape_next_cam0**_b** – 5 vids rotating anti-clockwise. So current shape on cam1 doesnt help. In _b, I will rotate for longer than 4 turns so cam0 itself will recognize repeating pattern and maybe predict. *[probably wont use at all as they are useless with multi-view (realised while recording)]*
- For each video model picks from ['square','rectangle','circle','triangle'] as next shape on cam0.
- Caveat: This experiment not designed as meticulously like the shellgame-multiview. 

### Workflow for multiview eval
Compress cam0 and cam1 down to 480p-20fps --> Stitch both views together --> Write custom prompt --> API call

## Result:
- All api calls ran smoothly, quickly and with much fewer tokens (granted the videos are short at ~5s)
- Perfect prediction on ALL 16 videos (both _a and _b experiments). **top1acc = 1.0**
- 11vids of _a that were correct show that model understands whats is happening. (e.g. 261125-130240 has only 3 turns)

In [1]:
## conda env vgd has openai
import base64
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
import numpy as np
from tqdm import tqdm

# import cv2

In [2]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

# Preparing main index_df with vqa videos

In [3]:
## index_df
index_df = pd.read_csv('/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/index.csv')
print(index_df.shape)

index_df = index_df[index_df['session'].str.startswith('which_shape_next_cam0')].reset_index(drop=True)
print(index_df.shape)

index_df['cam'] = 'cam0'
index_df = index_df.rename(columns={'id':'video_id'})

print(index_df.shape)
index_df.head(2)

(285, 4)
(16, 4)
(16, 5)


,video_id,template,placeholders,session,cam
0,261125-125558,square,ORIGAMI_4_SIDES,which_shape_next_cam0_a,cam0
1,261125-125633,triangle,ORIGAMI_4_SIDES,which_shape_next_cam0_a,cam0


In [4]:
labels_list = index_df.template.unique().tolist()
labels_list.sort()

index_df.template.value_counts()

template
square       5
triangle     4
rectangle    4
circle       3
Name: count, dtype: int64

In [5]:
index_df.session.value_counts()

session
which_shape_next_cam0_a    11
which_shape_next_cam0_b     5
Name: count, dtype: int64

### select videos

In [6]:
for i, row in index_df.iterrows():
    print(f"'{row['video_id']}', # --> {row['template']:5} -- {row['session']} ")


'261125-125558', # --> square -- which_shape_next_cam0_a 
'261125-125633', # --> triangle -- which_shape_next_cam0_a 
'261125-125913', # --> rectangle -- which_shape_next_cam0_a 
'261125-130240', # --> circle -- which_shape_next_cam0_a 
'261125-130305', # --> triangle -- which_shape_next_cam0_a 
'261125-130345', # --> square -- which_shape_next_cam0_a 
'261125-130412', # --> rectangle -- which_shape_next_cam0_b 
'261125-130439', # --> rectangle -- which_shape_next_cam0_b 
'261125-130458', # --> triangle -- which_shape_next_cam0_b 
'261125-130522', # --> circle -- which_shape_next_cam0_b 
'261125-130550', # --> rectangle -- which_shape_next_cam0_b 
'261125-131157', # --> square -- which_shape_next_cam0_a 
'261125-131216', # --> circle -- which_shape_next_cam0_a 
'261125-131241', # --> square -- which_shape_next_cam0_a 
'261125-131306', # --> triangle -- which_shape_next_cam0_a 
'261125-131326', # --> square -- which_shape_next_cam0_a 


In [7]:
vids_list = [
'261125-125558', # --> square -- which_shape_next_cam0_a 
'261125-125633', # --> triangle -- which_shape_next_cam0_a 
'261125-125913', # --> rectangle -- which_shape_next_cam0_a 
'261125-130240', # --> circle -- which_shape_next_cam0_a 
'261125-130305', # --> triangle -- which_shape_next_cam0_a 
'261125-130345', # --> square -- which_shape_next_cam0_a 
'261125-130412', # --> rectangle -- which_shape_next_cam0_b 
'261125-130439', # --> rectangle -- which_shape_next_cam0_b 
'261125-130458', # --> triangle -- which_shape_next_cam0_b 
'261125-130522', # --> circle -- which_shape_next_cam0_b 
'261125-130550', # --> rectangle -- which_shape_next_cam0_b 
'261125-131157', # --> square -- which_shape_next_cam0_a 
'261125-131216', # --> circle -- which_shape_next_cam0_a 
'261125-131241', # --> square -- which_shape_next_cam0_a 
'261125-131306', # --> triangle -- which_shape_next_cam0_a 
'261125-131326', # --> square -- which_shape_next_cam0_a 
]
print(len(vids_list))

16


In [8]:
print(index_df.shape)
index_df = index_df[index_df.video_id.isin(vids_list)]
print(index_df.shape)

(16, 5)
(16, 5)


In [13]:
import shutil
# ------------- chunk to copy videos ------------- #
source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/"
dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/zz_temp_dump/"

# 2. Clear destination directory if it exists, then recreate it
if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)
os.makedirs(dest_dir, exist_ok=True)

# 3. Get unique video IDs and copy matching files
unique_video_ids = vids_list

for filename in os.listdir(source_dir):
    if any(video_id in filename for video_id in unique_video_ids):
        shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
# ------------------------------------------------ #

# VLM Evaluation 

### Compress Videos. 9second videos too long and high def. api call can handle <10MB vids and mine are well over 15mb per single view

#### 1. compress to lower res and store as mp4 (both cam0 and cam1)
#### 2. Concatenate the 2 views per video carefully - Because API cannot ingest 2 separate videos at the same time --> combined video for multiview evaluation

In [ ]:
import os
import subprocess
from tqdm import tqdm

raw_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/raw"
compressed_dir = "data/compressed_videos"

os.makedirs(compressed_dir, exist_ok=True)

pbar = tqdm(vids_list, desc="Compressing Videos")
for vid in pbar:
    # Strip extension if vids_list already includes '.avi'
    stem = os.path.splitext(vid)[0]
    
    input_path = os.path.join(raw_dir, f"{stem}_cam1.avi")       # Change to f"{stem}_cam0.avi" if suffix is still used
    output_path = os.path.join(compressed_dir, f"{stem}_cam1.mp4")

    if os.path.exists(output_path):
        print(f"[-] Skipping (already exists): {stem}_cam1.mp4")
        continue

    if not os.path.exists(input_path):
        print(f"[!] Warning (missing input): {input_path}")
        continue

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-loglevel", "error",
            "-i", input_path,
            "-vf", "fps=20,scale=-2:480p",
            "-c:v", "libx264",
            "-crf", "23",
            "-an",
            output_path,
        ],
        check=True,
    )

    orig_mb = os.path.getsize(input_path) / (1024 * 1024)
    new_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"[*] Converted {stem}: {orig_mb:.2f} MB -> {new_mb:.2f} MB")

In [ ]:
## 31mins total to compress all 41 cam0 videos to 480p 20fps mp4 format.
## now doing same videos cam1 with a tqdm progressbar.

In [ ]:
import cv2

def concatenate_videos_side_by_side(video_path_0, video_path_1, output_path):
    """Stitches two synchronized videos horizontally into a single video file."""
    if os.path.exists(output_path):
        print(f"[-] Skipping concatenation: '{output_path}' already exists.")
        return
    
    cap0 = cv2.VideoCapture(video_path_0)
    cap1 = cv2.VideoCapture(video_path_1)

    fps = cap0.get(cv2.CAP_PROP_FPS) or 30.0
    width0 = int(cap0.get(cv2.CAP_PROP_FRAME_WIDTH))
    height0 = int(cap0.get(cv2.CAP_PROP_FRAME_HEIGHT))
    height1 = int(cap1.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Match height of video 1 to video 0 if they differ
    target_height = height0

    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    # Combined width = width of cam0 + width of cam1
    out_width = width0 + int(cap1.get(cv2.CAP_PROP_FRAME_WIDTH) * (target_height / height1))
    out = cv2.VideoWriter(output_path, fourcc, fps, (out_width, target_height))

    while cap0.isOpened() and cap1.isOpened():
        ret0, frame0 = cap0.read()
        ret1, frame1 = cap1.read()

        if not ret0 or not ret1:
            break

        # Resize frame1 if heights don't match exactly
        if frame1.shape[0] != target_height:
            new_w = int(frame1.shape[1] * (target_height / frame1.shape[0]))
            frame1 = cv2.resize(frame1, (new_w, target_height))

        # Concatenate horizontally (Left: View 0 | Right: View 1)
        combined_frame = cv2.hconcat([frame0, frame1])
        out.write(combined_frame)

    cap0.release()
    cap1.release()
    out.release()

    print(f"[*] New file created: {output_path}")

In [ ]:
videos_file_path = "data/compressed_videos/"
combined_videos_file_path = 'data/combined_videos/'

pbar = tqdm(vids_list, desc="Combining Videos")
for video_name in pbar:
    path_cam0 = f"{videos_file_path}{video_name}_cam0.mp4"
    path_cam1 = f"{videos_file_path}{video_name}_cam1.mp4"
    combined_path = f"{combined_videos_file_path}{video_name}_combined.mp4"

    # Stitch the videos horizontally
    concatenate_videos_side_by_side(path_cam0, path_cam1, combined_path)

In [ ]:
## compressing cam0/cam1 AND combining videos: 3 loops ran in run.py

### Run one video

In [20]:
# Load environment variables from .env
load_dotenv()

True

In [13]:
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"), # openrouter change
    base_url="https://openrouter.ai/api/v1", # openrouter change
)
client

In [14]:
# Encode combined video to base64
video_name = vids_list[0]

video_file_path = f'data/combined_videos/{video_name}_combined.mp4'

with open(video_file_path, "rb") as f:
    b64_video = base64.b64encode(f.read()).decode("utf-8")
    b64_size_mb = len(b64_video) / (1024 ** 2)
    print(f"Base64 payload size: {b64_size_mb:.2f} MB")

Base64 payload size: 4.57 MB


In [15]:
len(b64_video)

4793400

In [16]:
assert not b64_video.startswith("data:"), "Duplicate data: prefix detected"
assert "\n" not in b64_video, "Newlines found in base64 string"

In [17]:
system_prompt = (
    "You are an expert multi-view 3D spatial-temporal reasoning and object-tracking model. "
    "You are provided with a single synchronized split-screen video showing two side-by-side camera views of the exact same physical event: "
    "the LEFT half is 'cam0' (a bird's-eye top-down camera looking straight down at the top face of a 4-sided rectangular paper prism), "
    "and the RIGHT half is 'cam1' (a table-level side camera looking horizontally at one vertical side face of the prism). "
    "Each of the four longitudinal sides of the prism has a unique geometric shape drawn on it. "
    "During the video, hands rotate the prism in consistent 90-degree turns around its longitudinal axis. "
    "You must synthesize spatial and temporal cues across both camera views to determine the 3D orientation of the prism, "
    "the direction of rotation, and which shape will appear on the top face ('cam0') after one additional 90-degree turn in the same direction."
)

user_prompt = """
Analyze this synchronized split-screen video ('cam0' bird's-eye top view on the left, 'cam1' table-level side view on the right) of a 4-sided paper prism being rotated, and predict which shape will appear on the top face in 'cam0' if the prism is rotated one more 90-degree turn in the exact same direction after the video ends.

Instructions:
1. Initial State: Identify the shape visible on the top face in 'cam0' and the shape visible on the side face in 'cam1' at the beginning of the video.
2. Rotation Tracking: Track every 90-degree rotation step chronologically across both views, noting how faces move between 'cam0' (top) and 'cam1' (side) to establish the exact direction of rotation and the prism's 3D face adjacency.
3. Final State at Video End: Identify the shape currently on the top face in 'cam0' and the shape currently on the side face in 'cam1' at the end of the recording.
4. Next-Turn Prediction: Extrapolate one additional 90-degree rotation in the exact same direction from the final state, and state the single lowercase name of the shape (e.g., "triangle", "square", "rectangle", "circle") that will appear on the top face in 'cam0'.
5. Metrics: Provide a confidence score (float between 0.0 and 1.0, where 1.0 is completely certain and 0.25 is random guessing among 4 sides), the exact integer number of video frames sampled and processed.

Output strictly in JSON matching this schema:
{
  "initial_cam0_shape": "<shape>",
  "initial_cam1_shape": "<shape>",
  "rotation_sequence": "start(cam0:square, cam1:rectangle) -> turn1(cam0:rectangle, cam1:triangle)",
  "final_cam0_shape": "<shape>",
  "final_cam1_shape": "<shape>",
  "predicted_next_cam0_shape": "<lowercase shape name>",
  "confidence": 0.0,
  "frames_seen": 60
}
"""

print(user_prompt)


Analyze this synchronized split-screen video ('cam0' bird's-eye top view on the left, 'cam1' table-level side view on the right) of a 4-sided paper prism being rotated, and predict which shape will appear on the top face in 'cam0' if the prism is rotated one more 90-degree turn in the exact same direction after the video ends.

Instructions:
1. Initial State: Identify the shape visible on the top face in 'cam0' and the shape visible on the side face in 'cam1' at the beginning of the video.
2. Rotation Tracking: Track every 90-degree rotation step chronologically across both views, noting how faces move between 'cam0' (top) and 'cam1' (side) to establish the exact direction of rotation and the prism's 3D face adjacency.
3. Final State at Video End: Identify the shape currently on the top face in 'cam0' and the shape currently on the side face in 'cam1' at the end of the recording.
4. Next-Turn Prediction: Extrapolate one additional 90-degree rotation in the exact same direction from th

In [18]:
prompt_english_words = len(system_prompt.split(' ')) + len(user_prompt.split(' '))
print(f"Words in both prompts: {prompt_english_words}")
print(f"Estimated Text Tokens: {prompt_english_words*1.333:.1f}")

Words in both prompts: 404
Estimated Text Tokens: 538.5


In [19]:
start_time = time.time()

response = client.chat.completions.create(
    model="google/gemini-3.8-flash",
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {
                    "type": "video_url",
                    "video_url": {
                        "url": f"data:video/mp4;base64,{b64_video}",
                        "processing": "agentic",
                        },
                },
                {
                    "type": "text",
                    "text": user_prompt,
                },

            ],
        },
    ],
    extra_body={
        "reasoning": {"effort": "medium"},
    },
    response_format={"type": "json_object"},
    temperature=0.1,
)

print("Prompt tokens:", response.usage.prompt_tokens)
print("Completion tokens:", response.usage.completion_tokens)
print(f"Cost: $ {response.usage.cost:.6f}")

elapsed_time = round(time.time() - start_time, 2)

print(f"Elapsed time: {elapsed_time}s")

Prompt tokens: 1673
Completion tokens: 1463
Cost: $ 0.006741
Elapsed time: 15.85s


In [20]:
response.choices[0].message.content

'{\n  "initial_cam0_shape": "square",\n  "initial_cam1_shape": "rectangle",\n  "rotation_sequence": "start(cam0:square, cam1:rectangle) -> turn1(cam0:rectangle, cam1:triangle) -> turn2(cam0:triangle, cam1:circle) -> turn3(cam0:circle, cam1:square)",\n  "final_cam0_shape": "circle",\n  "final_cam1_shape": "square",\n  "predicted_next_cam0_shape": "square",\n  "confidence": 1.0,\n  "frames_seen": 15\n}'

In [21]:
prediction_json = json.loads(response.choices[0].message.content)
prediction_json

{'initial_cam0_shape': 'square',
 'initial_cam1_shape': 'rectangle',
 'rotation_sequence': 'start(cam0:square, cam1:rectangle) -> turn1(cam0:rectangle, cam1:triangle) -> turn2(cam0:triangle, cam1:circle) -> turn3(cam0:circle, cam1:square)',
 'final_cam0_shape': 'circle',
 'final_cam1_shape': 'square',
 'predicted_next_cam0_shape': 'square',
 'confidence': 1.0,
 'frames_seen': 15}

In [19]:
response.usage

CompletionUsage(completion_tokens=6553, prompt_tokens=45207, total_tokens=51760, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=6085, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=38595, image_tokens=None, text_tokens=None, video_tokens=2040), cost=0.032427375, is_byok=False, cost_details={'upstream_inference_cost': 0.032427375, 'upstream_inference_prompt_cost': 0.007853625, 'upstream_inference_completions_cost': 0.02457375})

In [22]:
# Extract JSON prediction content
prediction_json = json.loads(response.choices[0].message.content)

# Extract token & performance metadata
usage = response.usage

# 3. Update the dictionary to log both video file names

## this try except is from my testing when i wasnt giving the user_prompt to the models and model was returning confidence like 'low','high'
# try: 
#     conf = float(prediction_json.get("confidence", 0.0))
# except ValueError:
#     conf = 0.0

run_result = {
    "video_file": os.path.basename(video_file_path),
    "model": response.model,
    "elapsed_seconds": elapsed_time,
    "prompt_tokens": usage.prompt_tokens if usage else None,
    "completion_tokens": usage.completion_tokens if usage else None,
    "total_tokens": usage.total_tokens if usage else None,
    "cost": usage.cost if usage else None,

    'initial_cam0_shape': prediction_json.get("initial_cam0_shape"),
    'initial_cam1_shape': prediction_json.get("initial_cam1_shape"),
    'rotation_sequence': prediction_json.get("rotation_sequence"),
    'final_cam0_shape': prediction_json.get("final_cam0_shape"),
    'final_cam1_shape': prediction_json.get("final_cam1_shape"),
    'predicted_next_cam0_shape': prediction_json.get("predicted_next_cam0_shape"),
    "confidence": prediction_json.get("confidence"),
    "frames_seen": prediction_json.get("frames_seen"),
}

# Append result as a single JSON line
with open('data/vlm_eval_results/shellgame_predictions_testing_models.jsonl', "a", encoding="utf-8") as f:
    f.write(json.dumps(run_result) + "\n")

print(json.dumps(run_result, indent=2))

{
  "video_file": "261125-125558_combined.mp4",
  "model": "google/gemini-3.8-flash",
  "elapsed_seconds": 15.85,
  "prompt_tokens": 1673,
  "completion_tokens": 1463,
  "total_tokens": 3136,
  "cost": 0.006741,
  "initial_cam0_shape": "square",
  "initial_cam1_shape": "rectangle",
  "rotation_sequence": "start(cam0:square, cam1:rectangle) -> turn1(cam0:rectangle, cam1:triangle) -> turn2(cam0:triangle, cam1:circle) -> turn3(cam0:circle, cam1:square)",
  "final_cam0_shape": "circle",
  "final_cam1_shape": "square",
  "predicted_next_cam0_shape": "square",
  "confidence": 1.0,
  "frames_seen": 15
}


In [23]:
# 1. Load jsonl file directly into a DataFrame
df_testing_models = pd.read_json("data/vlm_eval_results/shellgame_predictions_testing_models.jsonl", lines=True)
pd.set_option('display.float_format', '{:,.3f}'.format)
print(df_testing_models.shape)
print(df_testing_models)
print(f"\nTotal Cost of Testing: $ {round(sum(df_testing_models.cost), 3)}")

(1, 15)
                   video_file                    model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens  cost initial_cam0_shape initial_cam1_shape                                                                                                                                   rotation_sequence final_cam0_shape final_cam1_shape predicted_next_cam0_shape  confidence  frames_seen
0  261125-125558_combined.mp4  google/gemini-3.8-flash           15.850           1673               1463          3136 0.007             square          rectangle  start(cam0:square, cam1:rectangle) -> turn1(cam0:rectangle, cam1:triangle) -> turn2(cam0:triangle, cam1:circle) -> turn3(cam0:circle, cam1:square)           circle           square                    square           1           15

Total Cost of Testing: $ 0.007


## Main Experiment: For loop over many videos

In [ ]:
## run_openrouter.py ranin tmux shell

# Loading and analyzing the jsonl outputs of the VLM
## Main Experiment: shellgame 
### gemini-3.8-flash (processing:'agentic')

In [9]:
# 1. Load jsonl file directly into a DataFrame
df = pd.read_json("data/vlm_eval_results/shellgame_predictions_gemini-3.8-flash-agentic.jsonl", lines=True)

print(df.shape)
print(df.head(10))

(16, 15)
                   video_file                    model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens      cost initial_cam0_shape initial_cam1_shape                                                                                                                                                                                                                                                      rotation_sequence final_cam0_shape final_cam1_shape predicted_next_cam0_shape  confidence  frames_seen
0  261125-125558_combined.mp4  google/gemini-3.8-flash            15.85           1673               1463          3136  6.74e-03             square          rectangle                                                                                                                     start(cam0:square, cam1:rectangle) -> turn1(cam0:rectangle, cam1:triangle) -> turn2(cam0:triangle, cam1:circle) -> turn3(cam0:circle, cam1:square)           circle           square           

In [10]:
index_df['video_file'] = index_df.video_id + '_combined.mp4'
print(index_df.shape)

df = pd.merge(df, index_df, how = 'left', on = ['video_file']) # LEFT JOIN
print(df.shape)
print(df.sample(3))

(16, 6)
(16, 20)
                   video_file                    model  elapsed_seconds  prompt_tokens  completion_tokens  total_tokens      cost initial_cam0_shape initial_cam1_shape                                                                                                                                   rotation_sequence final_cam0_shape final_cam1_shape predicted_next_cam0_shape  confidence  frames_seen       video_id  template     placeholders                  session   cam
0  261125-125558_combined.mp4  google/gemini-3.8-flash            15.85           1673               1463          3136  6.74e-03             square          rectangle  start(cam0:square, cam1:rectangle) -> turn1(cam0:rectangle, cam1:triangle) -> turn2(cam0:triangle, cam1:circle) -> turn3(cam0:circle, cam1:square)           circle           square                    square           1           15  261125-125558    square  ORIGAMI_4_SIDES  which_shape_next_cam0_a  cam0
3  261125-130240_combined.mp4  goog

In [12]:
df['correct_top1'] = df['predicted_next_cam0_shape'] == df['template']

print(f'Examples: {df.shape[0]}')
print(f'correct: {df.correct_top1.sum():4}')
print(f'top1acc: {df.correct_top1.mean():4}')

Examples: 16
correct:   16
top1acc:  1.0


In [14]:
df.columns

Index(['video_file', 'model', 'elapsed_seconds', 'prompt_tokens', 'completion_tokens', 'total_tokens', 'cost', 'initial_cam0_shape', 'initial_cam1_shape', 'rotation_sequence', 'final_cam0_shape', 'final_cam1_shape', 'predicted_next_cam0_shape', 'confidence', 'frames_seen', 'video_id', 'template', 'placeholders', 'session', 'cam', 'correct_top1'], dtype='str')

In [19]:
import textwrap
import pandas as pd

pd.set_option('display.float_format', '{:,.3f}'.format)

def analyze_df(df):
    print('=' * 20, 'DF Stats by Session', '=' * 20)

    stats = df.groupby(['session']).agg(
        videos=('video_file', 'count'),
        top1_acc=('correct_top1', 'mean'),
        avg_confidence=('confidence', 'mean'),
        avg_prompt_tokens=('prompt_tokens', 'mean'),
        avg_completion_tokens=('completion_tokens', 'mean'),
        total_cost_usd=('cost', 'sum'),
        avg_frames_seen=('frames_seen', 'mean'),
        sd_frames_seen=('frames_seen', 'std')
    )
    print(stats)
    print('\n')

    print('=' * 30, 'Individual Videos by session', '=' * 30)

    for session in df['session'].unique():
        print('\n' + '=' * 30, session, '=' * 30)

        for i, row in df[df['session'] == session].iterrows():
            # Include ground-truth column if present in your merged DataFrame
            summary_cols = [
                col for col in [
                    'video_id', 'session',
                    'template', 
                     'initial_cam0_shape', 'initial_cam1_shape', 'final_cam0_shape', 'final_cam1_shape', 'predicted_next_cam0_shape',
                       'correct_top1','frames_seen'
                ] if col in df.columns
            ]
            print(row[summary_cols].to_frame().T.to_string(index=False))
            print('\nRotation Sequence: ', textwrap.fill(str(row['rotation_sequence']), width=160))
            print('-' * 60)

In [20]:
analyze_df(df)

==================== DF Stats by Session ====================
                         videos  top1_acc  avg_confidence  avg_prompt_tokens  avg_completion_tokens  total_cost_usd  avg_frames_seen  sd_frames_seen
session                                                                                                                                             
which_shape_next_cam0_a      11     1.000           1.000          1,712.091              1,620.818           0.080           17.000          12.673
which_shape_next_cam0_b       5     1.000           1.000          3,073.600              1,870.000           0.041           23.600          11.929


============================== Individual Videos by session ==============================

============================== which_shape_next_cam0_a ==============================
     video_id                 session template initial_cam0_shape initial_cam1_shape final_cam0_shape final_cam1_shape predicted_next_cam0_shape correct_top1 fram